In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
DATA_DIR = Path("../data/raw")

customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
geolocation = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(
    DATA_DIR / "product_category_name_translation.csv"
)

In [ ]:
# ORDERS CLEANING

orders_clean = orders.copy()

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_clean[date_cols] = orders_clean[date_cols].apply(
    pd.to_datetime,
    errors="coerce"
)

# Data-quality flags
orders_clean["delivered_missing_delivery_date"] = (
    (orders_clean["order_status"] == "delivered") &
    (orders_clean["order_delivered_customer_date"].isna())
)

orders_clean["canceled_but_delivered"] = (
    (orders_clean["order_status"] == "canceled") &
    (orders_clean["order_delivered_customer_date"].notna())
)

orders_clean["delivered_missing_approval"] = (
    (orders_clean["order_status"] == "delivered") &
    (orders_clean["order_approved_at"].isna())
)

print("Orders cleaned:", orders_clean.shape)
print(
    "Delivered without delivery timestamp:",
    orders_clean["delivered_missing_delivery_date"].sum()
)

In [ ]:
# PRODUCTS CLEANING

products_clean = products.copy()

# Fix column-name typos
products_clean = products_clean.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})

# Zero grams is not a valid physical product weight
products_clean.loc[
    products_clean["product_weight_g"] == 0,
    "product_weight_g"
] = np.nan

# Missing category will later appear as Unknown
products_clean["product_category_name"] = (
    products_clean["product_category_name"]
    .fillna("unknown")
)

print("Products cleaned:", products_clean.shape)
print(
    "Missing weights:",
    products_clean["product_weight_g"].isna().sum()
)

In [ ]:
# PAYMENTS CLEANING

payments_clean = payments.copy()

payments_clean["payment_metadata_anomaly"] = (
    (payments_clean["payment_installments"] == 0) |
    (payments_clean["payment_type"] == "not_defined")
)

payments_clean["zero_value_payment"] = (
    payments_clean["payment_value"] == 0
)

print("Payments cleaned:", payments_clean.shape)
print(
    "Metadata anomalies:",
    payments_clean["payment_metadata_anomaly"].sum()
)

In [ ]:
# ORDER ITEMS CLEANING

order_items_clean = order_items.copy()

order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"],
    errors="coerce"
)

print("Order items cleaned:", order_items_clean.shape)

In [ ]:
# CUSTOMERS CLEANING

customers_clean = customers.copy()

customers_clean["customer_city"] = (
    customers_clean["customer_city"]
    .str.strip()
    .str.lower()
)

customers_clean["customer_state"] = (
    customers_clean["customer_state"]
    .str.strip()
    .str.upper()
)

print("Customers cleaned:", customers_clean.shape)

In [ ]:
# SELLERS CLEANING

sellers_clean = sellers.copy()

sellers_clean["seller_city"] = (
    sellers_clean["seller_city"]
    .str.strip()
    .str.lower()
)

sellers_clean["seller_state"] = (
    sellers_clean["seller_state"]
    .str.strip()
    .str.upper()
)

print("Sellers cleaned:", sellers_clean.shape)
print("Missing values:", sellers_clean.isnull().sum().sum())
print("Duplicate seller IDs:", sellers_clean["seller_id"].duplicated().sum())

In [ ]:
# CATEGORY TRANSLATION CLEANING

category_translation_clean = category_translation.copy()

missing_category_translations = pd.DataFrame({
    "product_category_name": [
        "portateis_cozinha_e_preparadores_de_alimentos",
        "pc_gamer",
        "unknown"
    ],
    "product_category_name_english": [
        "kitchen_portables_and_food_preparers",
        "gaming_pc",
        "unknown"
    ]
})

category_translation_clean = pd.concat(
    [
        category_translation_clean,
        missing_category_translations
    ],
    ignore_index=True
)

category_translation_clean = (
    category_translation_clean
    .drop_duplicates(subset="product_category_name")
)

print(
    "Translation categories:",
    len(category_translation_clean)
)

In [ ]:
# REVIEWS CLEANING

reviews_clean = reviews.copy()

reviews_clean["review_creation_date"] = pd.to_datetime(
    reviews_clean["review_creation_date"],
    errors="coerce"
)

reviews_clean["review_answer_timestamp"] = pd.to_datetime(
    reviews_clean["review_answer_timestamp"],
    errors="coerce"
)

# Useful flags
reviews_clean["has_comment"] = (
    reviews_clean["review_comment_message"].notna()
)

reviews_clean["has_title"] = (
    reviews_clean["review_comment_title"].notna()
)

# One-row-per-order review summary
reviews_order_clean = (
    reviews_clean
    .groupby("order_id")
    .agg(
        review_count=("review_id", "size"),
        review_score_mean=("review_score", "mean"),
        review_score_min=("review_score", "min"),
        review_score_max=("review_score", "max")
    )
    .reset_index()
)

reviews_order_clean["multiple_reviews"] = (
    reviews_order_clean["review_count"] > 1
)

reviews_order_clean["conflicting_scores"] = (
    reviews_order_clean["review_score_min"]
    != reviews_order_clean["review_score_max"]
)

print("Original review records:", len(reviews_clean))
print("Order-level reviews:", len(reviews_order_clean))
print(
    "Orders with multiple reviews:",
    reviews_order_clean["multiple_reviews"].sum()
)
print(
    "Orders with conflicting review scores:",
    reviews_order_clean["conflicting_scores"].sum()
)

In [ ]:
# GEOLOCATION CLEANING

geolocation_clean = geolocation.copy()

# Remove exact duplicate rows
geolocation_clean = (
    geolocation_clean
    .drop_duplicates()
    .copy()
)

# Keep coordinates within broad Brazilian geographic bounds
valid_brazil_coordinates = (
    geolocation_clean["geolocation_lat"].between(-35, 6)
    &
    geolocation_clean["geolocation_lng"].between(-75, -32)
)

geolocation_clean = geolocation_clean[
    valid_brazil_coordinates
].copy()

# Create one representative coordinate per ZIP prefix
geolocation_zip_clean = (
    geolocation_clean
    .groupby("geolocation_zip_code_prefix")
    .agg(
        geolocation_lat=("geolocation_lat", "median"),
        geolocation_lng=("geolocation_lng", "median")
    )
    .reset_index()
)

print(
    "Clean geolocation observations:",
    len(geolocation_clean)
)

print(
    "Unique ZIP locations:",
    len(geolocation_zip_clean)
)

In [ ]:
# FINAL CLEANING VALIDATION

print("Customers:", customers_clean.shape)
print("Orders:", orders_clean.shape)
print("Order items:", order_items_clean.shape)
print("Payments:", payments_clean.shape)
print("Reviews:", reviews_clean.shape)
print("Order-level reviews:", reviews_order_clean.shape)
print("Products:", products_clean.shape)
print("Sellers:", sellers_clean.shape)
print("Category translations:", category_translation_clean.shape)
print("Geolocation ZIPs:", geolocation_zip_clean.shape)

print("\nPrimary key checks")

print(
    "Duplicate customers:",
    customers_clean["customer_id"].duplicated().sum()
)

print(
    "Duplicate orders:",
    orders_clean["order_id"].duplicated().sum()
)

print(
    "Duplicate products:",
    products_clean["product_id"].duplicated().sum()
)

print(
    "Duplicate sellers:",
    sellers_clean["seller_id"].duplicated().sum()
)

print(
    "Duplicate geo ZIPs:",
    geolocation_zip_clean[
        "geolocation_zip_code_prefix"
    ].duplicated().sum()
)

In [ ]:
# SAVE PROCESSED DATASETS

OUTPUT_DIR = Path("../data/processed")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

customers_clean.to_csv(
    OUTPUT_DIR / "customers_clean.csv",
    index=False
)

orders_clean.to_csv(
    OUTPUT_DIR / "orders_clean.csv",
    index=False
)

order_items_clean.to_csv(
    OUTPUT_DIR / "order_items_clean.csv",
    index=False
)

payments_clean.to_csv(
    OUTPUT_DIR / "payments_clean.csv",
    index=False
)

reviews_clean.to_csv(
    OUTPUT_DIR / "reviews_clean.csv",
    index=False
)

reviews_order_clean.to_csv(
    OUTPUT_DIR / "reviews_order_clean.csv",
    index=False
)

products_clean.to_csv(
    OUTPUT_DIR / "products_clean.csv",
    index=False
)

sellers_clean.to_csv(
    OUTPUT_DIR / "sellers_clean.csv",
    index=False
)

category_translation_clean.to_csv(
    OUTPUT_DIR / "category_translation_clean.csv",
    index=False
)

geolocation_zip_clean.to_csv(
    OUTPUT_DIR / "geolocation_zip_clean.csv",
    index=False
)

print("Processed datasets saved successfully.")